# Background sequence coverage

The sequencing coverage for each base is available in [folder]/08_mutation_identification/U00096.coverage.tab.

* Missing coverage evidence: if a region is deleted in a sample, the only reads that will map to it will be mis-mappings
    * Calculate the coverage around each detected variant
    * Calculate the expected coverage, by pooling across samples
    * Variants where total depth is depleted compared to what's expected are more likely to be errors
* Null model for variant calling: what's the probability of not observing a variant?
* The samples at timepoint EX24 (60,000 gen) and EW00 (WT) were taken in exponential phase, so it's possible to infer chromosome replication dynamics

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns

from tqdm import tqdm

pd.options.display.max_columns = None

#folder = Path('/home/antoine/Genomics/Escherichia/A. Variant calling/Coverage_files')
#cov_file_suffix = '08_mutation_identification/U00096.coverage.tab'

In [2]:
m = pd.read_csv('../../../datasets/genomics/03_variant_filtering/intermediate/m_disentangled.tsv', sep = '\t', keep_default_na = False)

m.head()

,name,pop,gen,pos,pos_end,ref,alt,freq,chrom,allele_depth,total_depth,vcf_pos,raw_ref,raw_alt,rep_len,rep_nt,delta,locus,type,qual,raw_pos,raw_freq,raw_type,read_pop,back_ref,back_alt,back_delta,collapsed_rows,max_f,origin,wave
0,EA12,A,12,1660,1663,C×,C×,0.0392,JX871397,2.0,51.0,1661,,C,3,C,1.0,GAAATCGA.CCC.GCCAGACAG,REP,2.0,1660,0.0392,REP,A,,C,1,1,0.0392,A,1
1,EA01,A,1,1782,1783,A,C,0.0209,JX871397,6.0,242.0,1781,A,C,2,.,0.0,CTTCGTGA.A.CACCACTCT,SNP,1.9,1782,0.0209,SNP,A,A,C,0,1,0.0209,A,1
2,EA04,A,4,2998,3001,,C,0.0298,JX871397,2.0,67.0,2999,,C,3,C,1.0,CGGAAATG.CCC.GTGCTGACG,REP,1.9,2998,0.0298,REP,A,,C,1,1,0.0298,A,1
3,EA04,A,4,3564,3568,,G,0.0294,JX871397,2.0,68.0,3566,,G,4,G,1.0,AGACCCGT.GGGG.CCAATGCGT,REP,1.9,3564,0.0294,REP,A,,G,1,1,0.0294,A,1
4,EA07,A,7,3619,3623,,A,0.0426,JX871397,2.0,47.0,3621,,A,4,A,1.0,TACCCTGC.AAAA.TACCCAGCT,REP,2.5,3619,0.0426,REP,A,,A,1,1,0.0426,A,1


In [3]:
# Drop the variants that never reach a minimum frequency (ignoring waves)
freq_thr = 0.1

m['max_f_full'] = m.groupby(['chrom', 'pos', 'ref', 'alt']).freq.transform('max')
m_freq = m[m.max_f_full >= freq_thr].reset_index(drop = True)

# Fill the frequency curves with zeros
For now there is only a row if a variant was detected.

Add a row with freq = zero for all the generations where the variant was not detected, to get complete frequency curves.

In [4]:
# For satellite experiment I and A, set generation to 0
m_freq.loc[m_freq['pop'].str.len() > 1, 'gen'] = 0

In [5]:
import itertools

# Get unique combinations of 'name' and (pos, ref, alt)
name_x_variant = m_freq[['pop', 'origin', 'chrom', 'pos', 'ref', 'alt', 'wave']].drop_duplicates()

# Get all 'gen' values
gens = pd.DataFrame({'gen': range(0, 25)})

# Create all combinations of the unique_combinations with gen values
product = pd.DataFrame(
    list(itertools.product(name_x_variant.itertuples(index = False, name = None), gens['gen'])),
    columns=['variant', 'gen']
)

# Split the 'combination' tuple into separate columns
product[['pop', 'origin', 'chrom', 'pos', 'ref', 'alt', 'wave']] = pd.DataFrame(product['variant'].tolist(), index = product.index)
product.drop(columns=['variant'], inplace = True)

product.head()

,gen,pop,origin,chrom,pos,ref,alt,wave
0,0,A,A,JX871397,4929,C,T,1
1,1,A,A,JX871397,4929,C,T,1
2,2,A,A,JX871397,4929,C,T,1
3,3,A,A,JX871397,4929,C,T,1
4,4,A,A,JX871397,4929,C,T,1


In [6]:
# Add back static information (doesn't change with gen)
info = m_freq.copy()

# Keep only one row per pop x origin x pos x ref x alt combination
info['orig_pos'] = info.pos == info.raw_pos
info.sort_values(['pop', 'chrom', 'pos', 'ref', 'alt', 'wave', 'orig_pos'], inplace = True) # ascending, and False < True, so row 0 below is the row where pos != raw_pos
info['row_number'] = info.groupby(['pop', 'chrom', 'pos', 'ref', 'alt', 'wave']).cumcount()
info = info[(info.row_number == 0)].reset_index(drop = True)

# Keep only the columns that don't change with gen
info = info[['pop', 'chrom', 'pos', 'pos_end', 'ref', 'alt', 'wave', 'rep_len', 'rep_nt', 'locus', 'type']]

In [7]:
# Add back dynamic information
dyna = m_freq.copy()
dyna.drop(columns = ['pos_end', 'rep_len', 'rep_nt', 'locus', 'type'], inplace = True)

# Add it to the product df
d = product.merge(info, how = 'left')

# Unobserved variants have zero frequency
d = d.merge(dyna, how = 'left').fillna({'freq': 0, 'allele_depth': 0})

In [8]:
# Fill NAs
d['delta'] = d.groupby(['pop', 'chrom', 'pos', 'ref', 'alt', 'wave']).delta.bfill().ffill()
d['read_pop'] = d.groupby(['pop', 'chrom', 'pos', 'ref', 'alt', 'wave']).read_pop.bfill().ffill()

d['name'] = np.where(d['pop'].str.len() > 1, d['pop'], 'E' + d['pop'] +  d['gen'].astype(str).str.zfill(2))
d['read_name'] = np.where(d['pop'].str.len() > 1, d['pop'], 'E' + d['read_pop'] + d['gen'].astype(str).str.zfill(2))

# Add coverage information
#d = d.merge(sub_c, how = 'left')
#d.loc[d.gen == 0, 'coverage'] = 0

In [9]:
# Remove the missing generations from X, Y and Z, as well as positive generations for the satellite experiments A and I
d = d[
    ~((d['pop'] == 'X') & (d['gen'] > 12)) &
    ~((d['pop'] == 'Y') & (d['gen'] > 13)) &
    ~((d['pop'] == 'Z') & (d['gen'] > 8)) &
    ~((d['pop'].str.len() > 1) & (d['gen'] != 0))
].reset_index(drop = True)

d.drop(columns = ['max_f', 'max_f_full'], inplace = True)

In [10]:
d.to_csv('../../../datasets/genomics/03_variant_filtering/intermediate/d_complete.tsv', sep = '\t', index = False)

# Find masked variants
Some fixed variants end up disappearing, because the region they are in is later deleted. This looks like a reversion to the WT allele, but it's not.

To find problematic cases of this:
* Take all variants that fix at some point
* For each of them, look at the coverage of that region in the last timepoint (24, i.e. 60k gen)

This step is just to find where the wasked variants are. They will be corrected in the next notebook.

**Note: the coverage files are pretty heavy, so I don't include them in this archive. To continue, skip this and use the provided endpoint_coverages.tsv file for the next notebook.**

In [11]:
breseq_output_folder = Path('../../../datasets/genomics/breseq_coverage/')  # breseq output folders (not included in this repository)

In [12]:
if not breseq_output_folder.exists():
    print('breseq coverage files not found: using the provided endpoint_coverages.tsv')
else:
    main_pops = list('ABCDEFGH')
    prev_filename = ''

    fixed_variants = d[(d['pop'].isin(main_pops)) & (d.gen <= 23) & (d.freq == 1)][['pop', 'chrom', 'pos']].drop_duplicates().sort_values(['pop', 'chrom', 'pos']).reset_index(drop = True)

    lines = []

    for n, i in tqdm(fixed_variants.iterrows(), total=fixed_variants.shape[0]):
    
        #if i['pop'] != 'A': break
        #if i['chrom'] != 'JX871397': break
    
        pop, chrom, pos = i[['pop', 'chrom', 'pos']]
        #print(pop, chrom, pos)
    
        # Coverage files, ex:
        # breseq_output_folder/AB/08_mutation_identification/JX871397.coverage.tab
        filename = breseq_output_folder / f'E{pop}24' / f'08_mutation_identification/{chrom}.coverage.tab'
    
        # Read the coverage file if it's a new one
        if filename != prev_filename:
        
            coverage_df = pd.read_csv(filename, sep = '\t')
            prev_filename = filename
    
        if len(coverage_df) > 0:
            coverage_row = coverage_df.iloc[pos - 1]

            # Check we have the right position
            assert coverage_row['position'] == pos

            unique_cov = int(coverage_row['unique_top_cov'] + coverage_row['unique_bot_cov'])
            redundant_cov = int(coverage_row['redundant_top_cov'] + coverage_row['redundant_bot_cov'])
        
        else:
            unique_cov, redundant_cov = 0, 0
    
        if unique_cov == 0:
            print(f'Masked variant in {pop}: {chrom} {pos}. Redundant coverage: {redundant_cov}.')
        
            # Add to record and warn
            lines.append([pop, chrom, pos, unique_cov, redundant_cov])
    
    # Make a DF
    endpoint_coverages = pd.DataFrame(lines, columns = ['pop', 'chrom', 'pos', 'endpoint_cov', 'endpoint_redundant_cov'])

breseq coverage files not found: using the provided endpoint_coverages.tsv


In [13]:
if breseq_output_folder.exists():
    endpoint_coverages.to_csv('../../../datasets/genomics/03_variant_filtering/intermediate/endpoint_coverages.tsv', sep = '\t', index = False)